# 关于model调用的invoke的使用

## 1、invoke的传参

### 1.1 文本的输入

举例：

In [ ]:
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
import os

# 从.env文件中加载环境变量
load_dotenv(override=True)

DASHSCOPE_API_KEY = os.getenv("DASHSCOPE_API_KEY")
DASHSCOPE_BASE_URL = os.getenv("DASHSCOPE_BASE_URL")

model = init_chat_model(
    model="qwen3.7-max",
    model_provider="openai",
    # temperature=1.5,
    api_key=DASHSCOPE_API_KEY,
    base_url=DASHSCOPE_BASE_URL,
    # max_tokens=10,
)
# 向模型发送单条数据
response = model.invoke("简单介绍一下你自己")
print(response.content)

In [ ]:
response = model.invoke("翻译以下的汉字：你好世界")
print(response)

## 1.2 字典列表

举例1：单句对话

In [ ]:
message = [
    {"role":"system","content":"你是一个电商客服"},
    {"role":"user","content":"我想退货退款"},
    {"role":"assistant","content":"AI回复"},

]

response = model.invoke(message)
print(response.content)

举例2：涉及多轮对话

In [ ]:
message = [
    {"role":"system","content":"你是一个电商客服"},
    {"role":"user","content":"我想退货退款"},
    {"role":"assistant","content":"AI回复"},
    {"role":"user","content":"我刚刚问你什么问题？"},

]

response = model.invoke(message)
print(response.content)

举例3：如果不传递历史，AI会“失忆”

In [ ]:
message1 = [
    {"role":"system","content":"你是一个聊天ai精灵" },
    {"role":"user","content":"你好，我是飘飘" },
]
# 第一次对话
response1 = model.invoke(message1)
# 打印响应
print(f"AI的回复1：{response1.content}")

message2 = [
    {"role":"user","content":"我叫什么名字？"}
]
# 第二次对话
response2 = model.invoke(message2)
# 打印响应
print(f"AI的回复2：{response2.content}")

作为对比，传递记忆

In [ ]:
conversation = [
    {"role":"system","content":"你是一个聊天ai精灵"},
    {"role":"user","content":"你好，我是飘飘"}
]

# 第一次对话
response1 = model.invoke(conversation)
# 打印响应
print(f"AI的回复1: {response1.content}")

# 添加记忆
conversation.append({"role":"assistant","content":response1.content})
conversation.append({"role":"user","content":"我叫什么名字？"})

# 第二次对话
response2 = model.invoke(conversation)
print(f"AI的回复2: {response2.content}")

### 1.3 消息对象列表

举例1：单句对话

In [ ]:
from langchain_core.messages import HumanMessage, SystemMessage

message = [
    SystemMessage(content="你是一个电商客服"),
    HumanMessage(content="我想退货退款"),
]

response = model.invoke(message)
print(response.content)

举例2：传递记忆

In [ ]:
from langchain_core.messages import HumanMessage, SystemMessage, AIMessage

conversation = [
    SystemMessage(content="你是一个智能聊天机器人"),
    HumanMessage(content="你好，我叫飘飘"),
]

# 第一轮对话
response1 = model.invoke(conversation)
# 打印响应
print(f"AI的回复1是：{response1.content}")

# 添加记忆
conversation.append(AIMessage(content=response1.content))
conversation.append(HumanMessage(content="我叫什么名字？"),)

# 第二轮对话
response2 = model.invoke(conversation)
print(f"AI的回复2是：{response2.content}")


## 2. invoke的返回值

举例1:

In [ ]:
response = model.invoke([HumanMessage(content="2 + 3 * 2 = ?")])

In [ ]:
print(type(response))

In [ ]:
print(response)

美化输出：

In [ ]:
from rich import print as rprint

rprint(response)

例如：

In [ ]:
from langchain_core.messages import AIMessage

res = AIMessage(
    # --- 核心内容 ---
    content='2 + 3 * 2 = **8**',  # 模型生成的最终文本答案
    additional_kwargs={
        'refusal': None# 模型拒绝回答的情况（如触碰安全策略），None 表示正常回答
    },

    # --- 响应元数据（API 返回的详细原始数据） ---
    response_metadata={
        'token_usage': {
            'com·pletion_tokens': 15,    # 生成回答消耗的 Token 数（输出）
            'prompt_tokens': 16,        # 用户输入消耗的 Token 数（输入）
            'total_tokens': 31,         # 本次交互总共消耗的 Token

            'completion_tokens_details': {
                'accepted_prediction_tokens': 0, # 预测性生成的 Token 数
                'audio_tokens': 0,               # 音频生成消耗（如有）
                'reasoning_tokens': 0,# 推理模型（如 o1）思考过程消耗的 Token
                'rejected_prediction_tokens': 0  # 被拒绝的预测 Token
            },

            'prompt_tokens_details': {
                'audio_tokens': 0,   # 输入中的音频 Token 数
                'cached_tokens': 0   # 命中的缓存 Token 数（能省钱/提速）
            },

            # --- 延迟性能监控（单位：毫秒 ms） ---
            'latency_checkpoint': {
                'engine_tbt_ms': 4,        # 引擎 Token 间平均间隔时间
                'engine_ttft_ms': 36,      # 引擎生成首个 Token 的时间
                'engine_ttlt_ms': 100,     # 引擎生成最后一个 Token 的时间
                'pre_inference_ms': 86, # 推理前的预处理耗时(安全审核、Token 化等预处理)
                'service_tbt_ms': 4, # 服务端token与token之间生成的间隔时间，决定了打字机效果是否丝滑。
                'service_ttft_ms': 280,  # 服务端接收到请求到输出首字的总时间
                'service_ttlt_ms': 338,    # 服务端完成全部输出的总时间
                'total_duration_ms': 259,  # 本次请求在系统中记录的总持续时长
                'user_visible_ttft_ms': 194   # 用户看到第一个字跳出来等待的时间
            }
        },
        'model_provider': 'openai',                  # 模型供应商
        'model_name': 'qwen3.7-max',    # 使用的具体模型版本
        'system_fingerprint': None,                  # 系统指纹，用于追踪模型后端的配置变更
        'id': 'chatcmpl-DgWobsxhDOqzjqVFwbZYKRnovpEiV', #API层面的响应 ID
        'service_tier': 'default',    # 服务层级（如按量付费或订阅）
        'finish_reason': 'stop',      # 停止原因：stop（自然结束）、length（长度受限）
        'logprobs': None              # 对数概率（通常用于分析词汇选择的可能性）
    },

    # --- LangChain 内部标识 ---
    id='lc_run--019e3659-5ee2-7b62-bc8a-741e27374b43-0',
    # LangChain 追踪此条运行的唯一 ID

    # --- 工具调用信息 ---
    tool_calls=[],          # 正常触发的外部工具调用列表
    invalid_tool_calls=[],  # 触发失败或格式错误的工具调用

    # --- 统一消耗元数据（LangChain 标准化后的消耗格式） ---
    usage_metadata={
        'input_tokens': 16,    # 输入 Token 数
        'output_tokens': 15,   # 输出 Token 数
        'total_tokens': 31,    # 总 Token 数
        'input_token_details': {
            'audio': 0,
            'cache_read': 0     # 从缓存中读取的输入数量
        },
        'output_token_details': {
            'audio': 0,
            'reasoning': 0      # 包含在输出中的推理 Token
        }
    }
)

举例2：

In [27]:
response = model.invoke("用一句话解释什么是 AI")
# 1. 获取回复内容
print("AI 回复:", response.content)
# 2. 获取响应元数据
metadata = response.response_metadata
print(f"使用的模型: {metadata['model_name']}")
print(f"结束原因: {metadata['finish_reason']}")
print(f"模型提供商：{metadata['model_provider']}\n")
# 3. 获取 Token 使用情况
usage = metadata.get('token_usage', {})
print(f"输入 tokens: {usage.get('prompt_tokens')}")
print(f"输出 tokens: {usage.get('completion_tokens')}")
print(f"总计 tokens: {usage.get('total_tokens')}")
# 4. 获取消息 ID
print(f"消息 ID: {response.id}")

AI 回复: AI（人工智能）是指让计算机或机器模拟人类的智能，使其能够像人一样感知、学习、推理和解决复杂问题的技术。
使用的模型: qwen3.7-max
结束原因: stop
模型提供商：openai

输入 tokens: 15
输出 tokens: 405
总计 tokens: 420
消息 ID: lc_run--019f329f-276e-7500-a069-15ada232aaa8-0
